In [ ]:
!mkdir -p Sources/01.04

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("\n🎉 Environment initialization process completed successfully!")

# HIXL 整体开发流程

学习了前面章节，你应该对 HIXL 的功能有了基本的了解，本节将以一个 H2rD 和 rD2H 数据传输的最小实现为例，带你熟悉 HIXL 的开发流程。

本节学习大纲如下：

- HIXL 应用开发流程
- HIXL 实现 H2rD 和 rD2H 数据传输样例


## 1. HIXL 应用开发流程

本节将简单介绍下 HIXL 的数据传输抽象模型和推理框架集成 HIXL 的方法，让你对 HIXL 应用整体开发流程上有整体的认知。

### 1.1 HIXL 端到端执行流程

HIXL 将端到端的数据传输流程抽象为 7 个步骤，为用户屏蔽了底层复杂的交互流程，降低了用户应用开发过程中使用单边通信能力的门槛。

<img src="./images/hixl_api_lifecycle.png" alt="HIXL API调用生命周期流程图" width="100%">

| 步骤 | 作用 | 对应接口 | 关键注意事项 |
| --- | --- | -- | --- |
| 1. 初始化 | 创建本端 HIXL Engine，准备运行时参数 | `Initialize` | 成功后必须和 `Finalize` 配对 |
| 2. 注册内存 | 声明可被 HIXL 使用的地址、长度和类型 | `RegisterMem` | 通常要在 `Connect` 前完成 |
| 3. 建链 | 与远端 Engine 建立通信连接 | `Connect` | 两端都需要完成初始化，远端标识要匹配 |
| 4. 传输 | 发起同步或异步读写 | `TransferSync` / `TransferAsync` | 地址必须合法，长度不能越界 |
| 5. 断链 | 停止继续访问远端链路 | `Disconnect` | 清理前先确保没有未完成传输 |
| 6. 解注册 | 释放注册内存句柄 | `DeregisterMem` | 先断链再解注册，避免远端仍访问该内存 |
| 7. 资源释放 | 释放 HIXL 全局资源 | `Finalize` | 不要和其他 HIXL 接口并发调用 |

### 1.2 框架集成指南

以大模型推理框架为例，要在昇腾 NPU 环境使用 HIXL 提供的高性能数据传输能力，通常要经过下面的步骤：

- 推理框架流程抽象：将推理框架抽象为“资源初始化模块”、“内存管理模块”、“模型推理模块”以及“资源释放模块”；
- 在“资源初始化模块”中执行步骤 1——HIXL 初始化；
- 在“内存管理模块”中执行步骤 2——将框架申请的内存注册到 HIXL 中；
- 在“模型推理模块”中执行步骤 3 和 4——完成和传输对端的建链并调用 HIXL 数据传输接口；
- 在“资源释放模块”中执行步骤 5、6、7——释放 HIXL 相关资源。


## 2. H2D 数据传输最小实现

本节以 A2 单节点场景下 H2rD 和 rD2H 数据传输为例，向你展现了上述 HIXL 的端到端执行流程的最小实现，让你对 HIXL 的执行流程有更加直观的感受。

样例使用 client-server 编程模式，其中 client 侧负责建链和发起数据传输请求。


### 2.1 Server 侧实现

server 侧主执行流程在 RunServer 函数中，执行流程如下：

初始化 --> 申请 Device 内存 --> 注册内存 --> 将内存地址保存在本地文件中 --> 等待数据传输完毕 --> 校验数据正确性 --> 内存解注册 --> 释放内存 --> 析构

> 为了简化流程，样例中 server 在单节点内通过文件的方式将内存地址暴露给 client 侧，实际业务过程中，也可通过 TCP 或者其他控制面消息实现，可参考 [hixl benchmark](https://gitcode.com/cann/hixl/blob/master/benchmarks/benchmark.cpp) 的实现，使用 TCP 进行内存地址发送和消息通知。


In [ ]:
%%writefile Sources/01.04/server.cpp

#include <chrono>
#include <cstdlib>
#include <fstream>
#include <iostream>
#include <thread>
#include "acl/acl_rt.h"
#include "hixl/hixl.h"

using namespace hixl;
namespace {
constexpr int32_t kWaitTransferSeconds = 3;
constexpr int32_t kInitialValue = 1;

void ServerFinalize(Hixl &engine, MemHandle handle, int32_t *device_buffer) {
  if (handle != nullptr) {
    Status hixl_ret = engine.DeregisterMem(handle);
    if (hixl_ret == SUCCESS) {
      std::cout << "[SERVER INFO] DeregisterMem success" << std::endl;
    } else {
      std::cout << "[SERVER ERROR] DeregisterMem failed, ret = " << hixl_ret << std::endl;
    }
  }
  if (device_buffer != nullptr) {
    aclError acl_ret = aclrtFree(device_buffer);
    if (acl_ret != ACL_SUCCESS) {
      std::cout << "[SERVER ERROR] aclrtFree failed, aclError = " << acl_ret << std::endl;
    }
  }
  engine.Finalize();
}

int32_t RunServer(const char *local_engine) {
  Hixl engine;
  int32_t host_value = kInitialValue;
  int32_t *device_buffer = nullptr;
  MemHandle handle = nullptr;

  Status hixl_ret = engine.Initialize(local_engine, {});
  if (hixl_ret != SUCCESS) {
    std::cout << "[SERVER ERROR] Initialize failed, ret = " << hixl_ret << std::endl;
    return -1;
  }
  std::cout << "[SERVER INFO] Initialize success" << std::endl;

  aclError acl_ret = aclrtMalloc(reinterpret_cast<void **>(&device_buffer), sizeof(int32_t), ACL_MEM_MALLOC_HUGE_ONLY);
  if (acl_ret != ACL_SUCCESS) {
    std::cout << "[SERVER ERROR] aclrtMalloc failed, aclError = " << acl_ret << std::endl;
    ServerFinalize(engine, handle, device_buffer);
    return -1;
  }
  acl_ret = aclrtMemcpy(device_buffer, sizeof(int32_t), &host_value, sizeof(int32_t), ACL_MEMCPY_HOST_TO_DEVICE);
  if (acl_ret != ACL_SUCCESS) {
    std::cout << "[SERVER ERROR] aclrtMemcpy H2D failed, aclError = " << acl_ret << std::endl;
    ServerFinalize(engine, handle, device_buffer);
    return -1;
  }

  MemDesc mem{reinterpret_cast<uintptr_t>(device_buffer), sizeof(int32_t)};
  hixl_ret = engine.RegisterMem(mem, MEM_DEVICE, handle);
  if (hixl_ret != SUCCESS) {
    std::cout << "[SERVER ERROR] RegisterMem failed, ret = " << hixl_ret << std::endl;
    ServerFinalize(engine, handle, device_buffer);
    return -1;
  }
  std::cout << "[SERVER INFO] RegisterMem success, server device address:" << device_buffer << std::endl;

  std::ofstream tmp_file("./tmp");
  if (!tmp_file) {
    std::cout << "[SERVER ERROR] Open ./tmp failed" << std::endl;
    ServerFinalize(engine, handle, device_buffer);
    return -1;
  }
  tmp_file << std::hex << reinterpret_cast<uintptr_t>(device_buffer) << std::endl;

  std::this_thread::sleep_for(std::chrono::seconds(kWaitTransferSeconds));
  acl_ret = aclrtMemcpy(&host_value, sizeof(int32_t), device_buffer, sizeof(int32_t), ACL_MEMCPY_DEVICE_TO_HOST);
  if (acl_ret != ACL_SUCCESS) {
    std::cout << "[SERVER ERROR] aclrtMemcpy D2H failed, aclError = " << acl_ret << std::endl;
    ServerFinalize(engine, handle, device_buffer);
    return -1;
  }
  std::cout << "[SERVER INFO] server value after transfer: " << host_value << std::endl;

  ServerFinalize(engine, handle, device_buffer);
  return 0;
}
}  // namespace

int main() {
  int32_t device_id = 0;
  aclError acl_ret = aclrtSetDevice(device_id);
  if (acl_ret != ACL_SUCCESS) {
    std::cout << "[SERVER ERROR] aclrtSetDevice failed, aclError = " << acl_ret << std::endl;
    return -1;
  }

  const auto &local_engine = "127.0.0.1:16000";
  const int32_t ret = RunServer(local_engine);

  (void)aclrtResetDevice(device_id);
  return ret;
}

### 2.2 Client 侧实现

client 侧主执行流程在 RunClient 函数中，执行流程如下：

初始化 --> 申请 Host 内存 --> 注册内存 --> 发起建链 --> 从本地文件读取 Server 侧内存地址 --> 发起数据传输 --> 发起断链 --> 内存解注册 --> 释放内存 --> 析构


In [ ]:
%%writefile Sources/01.04/client.cpp

#include <chrono>
#include <cstdlib>
#include <fstream>
#include <iostream>
#include <thread>
#include "acl/acl_rt.h"
#include "hixl/hixl.h"

using namespace hixl;
namespace {
constexpr int32_t kWaitRegSeconds = 1;
constexpr int32_t kWriteValue = 2;

void ClientFinalize(Hixl &engine, bool connected, const char *remote_engine, MemHandle handle, int32_t *src) {
  if (connected) {
    Status hixl_ret = engine.Disconnect(remote_engine);
    if (hixl_ret == SUCCESS) {
      std::cout << "[CLIENT INFO] Disconnect success" << std::endl;
    } else {
      std::cout << "[CLIENT ERROR] Disconnect failed, ret = " << hixl_ret << std::endl;
    }
  }
  if (handle != nullptr) {
    Status hixl_ret = engine.DeregisterMem(handle);
    if (hixl_ret == SUCCESS) {
      std::cout << "[CLIENT INFO] DeregisterMem success" << std::endl;
    } else {
      std::cout << "[CLIENT ERROR] DeregisterMem failed, ret = " << hixl_ret << std::endl;
    }
  }
  if (src != nullptr) {
    aclError acl_ret = aclrtFreeHost(src);
    if (acl_ret != ACL_SUCCESS) {
      std::cout << "[CLIENT ERROR] aclrtFreeHost failed, aclError = " << acl_ret << std::endl;
    }
  }
  engine.Finalize();
}

int32_t RunClient(const char *local_engine, const char *remote_engine) {
  Hixl engine;
  bool connected = false;
  int32_t *src = nullptr;
  MemHandle handle = nullptr;

  Status hixl_ret = engine.Initialize(local_engine, {});
  if (hixl_ret != SUCCESS) {
    std::cout << "[CLIENT ERROR] Initialize failed, ret = " << hixl_ret << std::endl;
    return -1;
  }
  std::cout << "[CLIENT INFO] Initialize success" << std::endl;

  aclError acl_ret = aclrtMallocHost(reinterpret_cast<void **>(&src), sizeof(int32_t));
  if (acl_ret != ACL_SUCCESS) {
    std::cout << "[CLIENT ERROR] aclrtMallocHost failed, aclError = " << acl_ret << std::endl;
    ClientFinalize(engine, connected, remote_engine, handle, src);
    return -1;
  }
  MemDesc mem{reinterpret_cast<uintptr_t>(src), sizeof(int32_t)};
  hixl_ret = engine.RegisterMem(mem, MEM_HOST, handle);
  if (hixl_ret != SUCCESS) {
    std::cout << "[CLIENT ERROR] RegisterMem failed, ret = " << hixl_ret << std::endl;
    ClientFinalize(engine, connected, remote_engine, handle, src);
    return -1;
  }
  std::cout << "[CLIENT INFO] RegisterMem success" << std::endl;

  std::this_thread::sleep_for(std::chrono::seconds(kWaitRegSeconds));
  hixl_ret = engine.Connect(remote_engine);
  if (hixl_ret != SUCCESS) {
    std::cout << "[CLIENT ERROR] Connect failed, ret = " << hixl_ret << std::endl;
    ClientFinalize(engine, connected, remote_engine, handle, src);
    return -1;
  }
  std::cout << "[CLIENT INFO] Connect success" << std::endl;
  connected = true;

  uintptr_t remote_addr = 0;
  std::ifstream tmp_file("./tmp");
  if (!(tmp_file >> std::hex >> remote_addr)) {
    std::cout << "[CLIENT ERROR] Read remote device address from ./tmp failed" << std::endl;
    ClientFinalize(engine, connected, remote_engine, handle, src);
    return -1;
  }

  TransferOpDesc op{reinterpret_cast<uintptr_t>(src), remote_addr, sizeof(int32_t)};
  hixl_ret = engine.TransferSync(remote_engine, READ, {op});
  if (hixl_ret != SUCCESS) {
    std::cout << "[CLIENT ERROR] TransferSync read failed, ret = " << hixl_ret << std::endl;
    ClientFinalize(engine, connected, remote_engine, handle, src);
    return -1;
  }
  std::cout << "[CLIENT INFO] TransferSync read success, read value from server:" << *src << std::endl;

  *src = kWriteValue;
  hixl_ret = engine.TransferSync(remote_engine, WRITE, {op});
  if (hixl_ret != SUCCESS) {
    std::cout << "[CLIENT ERROR] TransferSync write failed, ret = " << hixl_ret << std::endl;
    ClientFinalize(engine, connected, remote_engine, handle, src);
    return -1;
  }
  std::cout << "[CLIENT INFO] TransferSync write success, write value to server:" << *src << std::endl;

  ClientFinalize(engine, connected, remote_engine, handle, src);
  return 0;
}
}  // namespace

int main() {
  int32_t device_id = 1;
  aclError acl_ret = aclrtSetDevice(device_id);
  if (acl_ret != ACL_SUCCESS) {
    std::cout << "[CLIENT ERROR] aclrtSetDevice failed, aclError = " << acl_ret << std::endl;
    return -1;
  }

  const auto &local_engine = "127.0.0.1";
  const auto &remote_engine = "127.0.0.1:16000";
  const int32_t ret = RunClient(local_engine, remote_engine);

  (void)aclrtResetDevice(device_id);
  return ret;
}

### 2.3 Cmake 编译配置

使用 CMake 作为构建系统，其配置如下：


In [ ]:
%%writefile Sources/01.04/CMakeLists.txt

cmake_minimum_required(VERSION 3.16)

project(client_server_h2d LANGUAGES CXX)

if (DEFINED ENV{ASCEND_HOME_PATH})
    set(ASCEND_HOME_PATH $ENV{ASCEND_HOME_PATH})
else ()
    set(ASCEND_HOME_PATH /usr/local/Ascend/latest)
endif ()

function(hixl_h2d_builder target_name source_file)
    add_executable(${target_name} ${source_file})
    target_include_directories(${target_name} PRIVATE
            ${ASCEND_HOME_PATH}/include
    )
    target_compile_options(${target_name} PRIVATE
            -Wall
            -Werror
            -Wextra
    )

    target_link_directories(${target_name} PRIVATE
            ${ASCEND_HOME_PATH}/lib64
    )

    target_link_libraries(${target_name} PRIVATE
            cann_hixl
            acl_rt
            metadef
    )
endfunction()

hixl_h2d_builder(client client.cpp)
hixl_h2d_builder(server server.cpp)

### 2.4 编译和运行

将编译执行命令统一放在 run.sh 脚本中，通过执行脚本一键编译生成 server 和 client 可执行文件并执行。


In [ ]:
%%writefile Sources/01.04/run.sh

#!/bin/bash
set -e
rm -rf build
mkdir -p build
cd build
cmake .. && make

./server &
server_pid=$!

./client &
client_pid=$!

wait "${server_pid}"
wait "${client_pid}"

echo "Run success!"

再执行以下代码，编译并拉起 server 和 client 进程


In [ ]:
!cd Sources/01.04 && bash run.sh

## 课后练习

本节介绍了 HIXL 应用开发流程和 H2rD 和 rD2H 数据传输的最小实现，请根据学习内容完成以下题目进行自测。

1. （判断题）在 HIXL 端到端执行流程中，`RegisterMem` 通常应在 `Connect` 之前完成，否则建链后注册的内存不支持远端访问。

2. （判断题）在 HIXL 应用开发中，调用 `TransferSync` 进行数据传输时，源地址和目标地址必须合法，传输长度不能越界。

3. （单选题）以下哪个选项正确描述了 HIXL 端到端执行流程中初始化阶段的顺序？
    A. Initialize -> Connect -> RegisterMem
    B. Initialize -> RegisterMem -> Connect
    C. Connect -> Initialize -> RegisterMem
    D. RegisterMem -> Initialize -> Connect

4. （单选题）在 HIXL 资源释放流程中，以下哪种操作顺序是正确的？
    A. Finalize -> Disconnect -> DeregisterMem
    B. Disconnect -> Finalize -> DeregisterMem
    C. Disconnect -> DeregisterMem -> Finalize
    D. DeregisterMem -> Disconnect -> Finalize

5. （多选题）以下关于 HIXL 端到端执行流程的注意事项，哪些描述是正确的？
    A. Initialize 成功后必须和 Finalize 配对调用
    B. Disconnect 前应确保没有未完成的传输任务
    C. DeregisterMem 前应先 Disconnect，避免远端仍访问该内存
    D. Finalize 可以和其他 HIXL 接口并发调用

**执行以下代码获取答案。**


In [ ]:
!cat ./answer/01.04_answer.txt